In [58]:
import os
import time
from datetime import datetime
import pandas as pd
from dotenv import load_dotenv

In [59]:
from alpaca.data.historical.news import NewsClient
from alpaca.data.historical import StockHistoricalDataClient
from alpaca.data.requests import NewsRequest, StockBarsRequest
from alpaca.data.timeframe import TimeFrame
from alpaca.data.enums import Adjustment, DataFeed

In [60]:
load_dotenv("../.env")

API_KEY = os.getenv("API_KEY")
SECRET_KEY = os.getenv("SECRET_KEY")

In [94]:
news_client = NewsClient(API_KEY, SECRET_KEY)
price_client = StockHistoricalDataClient(API_KEY, SECRET_KEY)

In [95]:
# TODO: Pick Stocks
tech = ['AAPL']
healthcare = ["JNJ"]
...
TICKERS = tech + healthcare

In [96]:
# YEARS = range(2020, 2027) #consider pandemic years
# END_DATE  = datetime(2026, 10, 16)  # refer to alpaca, or datetime.now().date()
START_DATE = datetime(2020, 1, 1)
END_DATE = datetime(2026, 10, 16)
BENCHMARK = "SPY"

In [97]:
# Output folders
NEWS_DIR = "../data/raw/news"
PRICES_PATH = "../data/raw/prices.parquet"

os.makedirs(NEWS_DIR, exist_ok=True)

In [98]:
# func requsting one ticker in alpaca sdk (refer to alpaca sdk doc)
def fetch_news(symbol, start, end):

    request = NewsRequest(
        symbols=symbol,
        start=start,
        end=end,
        include_content=True,   # ask Alpaca to return the full article text
    )

    response = news_client.get_news(request)

    rows = []
    for article in response.data["news"]:
        rows.append({
            "ticker": symbol,
            "created_at": article.created_at,
            "headline": article.headline,
            "content": article.content,     # full text (HTML)
            "source": article.source,
            "symbols": article.symbols,
        })
    return pd.DataFrame(rows)

In [99]:
# Check: return a df of all news between start and end
test = fetch_news("AAPL", datetime(2026, 9, 17), datetime(2026, 10, 17))
test.head()

,ticker,created_at,headline,content,source,symbols
0,AAPL,2026-10-16 15:33:56+00:00,'Apple Set For UK Legal Fight Over iPhone Tap-...,,benzinga,"[AAPL, EWU]"
1,AAPL,2026-10-16 15:19:23+00:00,'PayPal And Epic Games Have Discussed Concerns...,,benzinga,"[AAPL, PYPL]"
2,AAPL,2026-10-16 13:14:35+00:00,Apple Stock Hits Record High as iPhone Duo Pre...,"<p><strong>Apple Inc.</strong> <a class=""ticke...",benzinga,[AAPL]
3,AAPL,2026-10-16 13:11:24+00:00,October Rate-Hike Odds Collapse as Warsh Keeps...,"<p>Despite surging inflation, Federal Reserve ...",benzinga,"[AAPL, SPY]"
4,AAPL,2026-10-16 12:38:48+00:00,"Apple’s $1,999 iPhone Duo Preorders Begin Toda...","<p><strong>Apple Inc.</strong> <a class=""ticke...",benzinga,"[AAPL, T, TMUS, VZ]"


In [100]:
print(test["created_at"].min(), test["created_at"].max())

2026-09-17 07:14:14+00:00 2026-10-16 15:33:56+00:00


In [101]:
# pull all tickers
for ticker in TICKERS:
    out_path = f"{NEWS_DIR}/{ticker}.parquet"

    # just get rid of file already exists
    if os.path.exists(out_path):
        print(f"{ticker}: already saved, skipping")
        continue

    df_ticker = fetch_news(ticker, START_DATE, END_DATE)
    df_ticker.to_parquet(out_path)
    print(f"{ticker}: {len(df_ticker)} articles saved")

AAPL: 23674 articles saved
JNJ: 4219 articles saved


In [102]:
# Check stock price - refer to alpaca_exploration

# request = StockBarsRequest(
#     symbol_or_symbols=ticker,      
#     timeframe=TimeFrame.Hour,        
#     start=start,                     
#     end=end,
#     feed=DataFeed.IEX,
# )

price_request = StockBarsRequest(
    symbol_or_symbols=TICKERS + [BENCHMARK],
    timeframe=TimeFrame.Day,
    start=datetime(2020, 1, 1),
    end=END_DATE,
    adjustment=Adjustment.ALL,
    feed=DataFeed.SIP,
)


In [103]:
price_client.get_stock_bars(price_request).df.head()

open   high    low  close       volume  \
symbol timestamp                                                            
AAPL   2020-01-02 05:00:00+00:00  71.28  72.33  71.03  72.27  146217352.0   
       2020-01-03 05:00:00+00:00  71.50  72.33  71.35  71.57  154711256.0   
       2020-01-06 05:00:00+00:00  70.69  72.18  70.44  72.14  125599968.0   
       2020-01-07 05:00:00+00:00  72.15  72.40  71.58  71.80  118227136.0   
       2020-01-08 05:00:00+00:00  71.50  73.26  71.50  72.95  142736804.0   

                                  trade_count   vwap  
symbol timestamp                                      
AAPL   2020-01-02 05:00:00+00:00     283489.0  71.84  
       2020-01-03 05:00:00+00:00     280441.0  71.91  
       2020-01-06 05:00:00+00:00     242868.0  71.66  
       2020-01-07 05:00:00+00:00     227871.0  71.96  
       2020-01-08 05:00:00+00:00     261773.0  72.50

In [104]:
prices = price_client.get_stock_bars(price_request).df.reset_index()
prices.head()

,symbol,timestamp,open,high,low,close,volume,trade_count,vwap
0,AAPL,2020-01-02 05:00:00+00:00,71.28,72.33,71.03,72.27,146217352.0,283489.0,71.84
1,AAPL,2020-01-03 05:00:00+00:00,71.50,72.33,71.35,71.57,154711256.0,280441.0,71.91
2,AAPL,2020-01-06 05:00:00+00:00,70.69,72.18,70.44,72.14,125599968.0,242868.0,71.66
3,AAPL,2020-01-07 05:00:00+00:00,72.15,72.40,71.58,71.80,118227136.0,227871.0,71.96
4,AAPL,2020-01-08 05:00:00+00:00,71.50,73.26,71.50,72.95,142736804.0,261773.0,72.50


In [105]:
prices.to_parquet(PRICES_PATH)

In [106]:
news_all = pd.concat(
    [pd.read_parquet(f"{NEWS_DIR}/{t}.parquet") for t in TICKERS],
    ignore_index=True,
)
print(len(news_all))

news_all.head()

27893


,ticker,created_at,headline,content,source,symbols
0,AAPL,2026-10-15 18:33:56+00:00,Anthropic Needs Nearly $2.7B in Additional Mon...,"<figure class=""wp-block-image size-large""><img...",benzinga,"[AAPL, AMZN, BTCUSD, GOOG, META, MSFT, NVDA, Q..."
1,AAPL,2026-10-15 18:04:31+00:00,Michael Burry's 'Red October' Warning: Why QQQ...,<p>Falling bond yields are failing to revive t...,benzinga,"[AAPL, AMZN, DIA, GOOGL, META, MSFT, NVDA, QQQ..."
2,AAPL,2026-10-15 17:26:44+00:00,"Meta Announces Release Of Muse Spark Open, The...",,benzinga,"[AAPL, META, MSFT]"
3,AAPL,2026-10-15 17:04:44+00:00,'Apple and Meta team up to make Muse AI model ...,,benzinga,"[AAPL, META, MSFT]"
4,AAPL,2026-10-15 16:47:46+00:00,Apple Hits $5 Trillion Market Cap as Nvidia's ...,"<p><strong>Apple Inc.</strong> <a href=""https:...",benzinga,"[AAPL, CRWV, NVDA, ORCL, TSM]"


In [107]:
# articles per ticket
news_all.groupby("ticker").size().sort_values()

ticker
JNJ      4219
AAPL    23674
dtype: int64

In [108]:
# articles per year
news_all["year"] = pd.to_datetime(news_all["created_at"]).dt.year
news_all.groupby(["ticker", "year"]).size().unstack()

year,2020,2021,2022,2023,2024,2025,2026
ticker,,,,,,,
AAPL,2035,3645,4246,4272,3873,3318,2285
JNJ,578,1161,575,574,500,510,321


In [109]:
# price coverage
prices.groupby("symbol")["timestamp"].agg(["min", "max", "count"]) #1706 trading days for now 

,min,max,count
symbol,,,
AAPL,2020-01-02 05:00:00+00:00,2026-10-15 04:00:00+00:00,1706
JNJ,2020-01-02 05:00:00+00:00,2026-10-15 04:00:00+00:00,1706
SPY,2020-01-02 05:00:00+00:00,2026-10-15 04:00:00+00:00,1706
